**Sample ID**: 72

**Query**:

I need help with an exchange.

**DB Type**: Base Case

**Case Description**:

The user is Ethan Garcia from Denver, CO 80280 (user ID: `ethan_garcia_1261`). He has three recent orders: #W4967593, #W9911714, and #W5733668. He wants to exchange his shoes with item ID 4107812777 and use a gift card to cover any possible charges. However, if the agent asks for confirmation, Ethan changes his mind and also wants to change the item under product ID 1656367028 to item ID 1421289881. Both the exchanges are concerned with the order #W9911714. He is not familiar with the domain and might confuse product and item IDs, so he asks the agent to figure out the details on its own if needed. Ethan also wants to know his gift card balance after all these actions.




```
<multiturn info>
User Identification: Ethan Garcia, ZIP 80280 (Information Gathering)
Location: Denver (Information Gathering)
Item and Action: Exchange shoes from order #W9911714 (Information Gathering)
Target Item: Exchange to item 4107812777 (Information Gathering)
Additional Exchange: If asked for confirmation, change the item under product 1656367028 to item 1421289881 (Iterative Refinement)
Payment Method: Use gift card to cover charges (Information Gathering)
Additional Request: Request gift card balance (Follow Up Request)
All exchanges are for order #W9911714 (Scope)
</multiturn info>

```

**Global/Context Variables:**

**APIs:**

- retail

# Set Up

## Download relevant files

In [ ]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.1 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.1.zip (ID: 1qDwJfS_gCyw6Z3HA1llw2KYZMwNHNHsi)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.1.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ Successfully generated 68 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [ ]:
!pip install -r /content/APIs/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 946.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 52.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 85.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.1/245.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.9/443.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.3/187.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.9 MB/s eta 0

## Import APIs and initiate DBs

In [ ]:
# Using default DB for the Tau Benchmark
import retail

retail.SimulationEngine.db.load_state("/content/DBs/RetailDefaultDB.json")


# Initial Assertion

1. A user named Ethan Garcia with ZIP code `80280` and ID `ethan_garcia_1261` exists.
2. The user has an orders with ID `#W9911714`.
3. The order with ID `#W9911714` contains an item with the name `shoe` whose ID is not `4107812777`.
4. The order with ID `#W9911714` contains a product with ID `1656367028`.


In [ ]:
import retail
import re
from Scripts.assertions_utils import *

# --- Constants ---
USER_FIRST_NAME = "Ethan"
USER_LAST_NAME = "Garcia"
USER_ZIP_CODE = "80280"
EXPECTED_USER_ID = "ethan_garcia_1261"
EXPECTED_ORDER_ID = "#W9911714"
EXPECTED_ITEM_KEYWORD = "shoe"
EXPECTED_PRODUCT_ID = "1656367028"
FORBIDDEN_ITEM = "4107812777"

# --- Data Gathering ---
user_id = None
user_details = None
api_error_user = None

try:
    user_id = retail.find_user_id_by_name_zip(
        first_name=USER_FIRST_NAME,
        last_name=USER_LAST_NAME,
        zip_code=USER_ZIP_CODE
    )
    if user_id:
        user_details = retail.get_user_details(user_id=user_id)
except Exception as e:
    api_error_user = str(e)

# --- Assertion 1: Verify user exists and has expected ID ---
assertion_condition_1 = (
    compare_strings(user_id, EXPECTED_USER_ID) and user_details is not None
)
assertion_message_1 = (
    f"Assertion 1 Failed: Expected user '{USER_FIRST_NAME} {USER_LAST_NAME}' with ZIP '{USER_ZIP_CODE}' "
    f"to have ID '{EXPECTED_USER_ID}'. Found user_id: {user_id}. API Error: {api_error_user}"
)
assert assertion_condition_1, assertion_message_1

# --- Assertion 2: Verify order #W9911714 exists in user's orders ---
user_orders = user_details.get("orders", []) if user_details else []
assertion_condition_2 = compare_is_list_subset(EXPECTED_ORDER_ID, user_orders)
assertion_message_2 = (
    f"Assertion 2 Failed: Expected order '{EXPECTED_ORDER_ID}' in user's orders. "
    f"User orders found: {user_orders}"
)
assert assertion_condition_2, assertion_message_2

# --- Retrieve order details ---
order_details = None
api_error_order = None

try:
    order_details = retail.get_order_details(order_id=EXPECTED_ORDER_ID)
except Exception as e:
    api_error_order = str(e)

# --- Assertion 3: Verify order contains an item with the name "shoe" ---
items = order_details.get("items", []) if order_details else []
item_names_and_ids = {item.get("item_id"): item.get("name", "") for item in items if item.get("item_id") != None}
has_shoe_item = any(re.search(re.escape(EXPECTED_ITEM_KEYWORD), item_name, re.IGNORECASE | re.DOTALL) and item_id != FORBIDDEN_ITEM for item_id, item_name in item_names_and_ids.items())
assertion_condition_3 = has_shoe_item
assertion_message_3 = (
    f"Assertion 3 Failed: Expected item with name containing '{EXPECTED_ITEM_KEYWORD}' in order items. "
    f"Found item names: {item_names_and_ids}"
)
assert assertion_condition_3, assertion_message_3

# --- Assertion 4: Verify order contains product with ID 1656367028 ---
product_ids = [item.get("product_id") for item in items]
has_expected_product = compare_is_list_subset(EXPECTED_PRODUCT_ID, product_ids)
assertion_condition_4 = has_expected_product
assertion_message_4 = (
    f"Assertion 4 Failed: Expected product ID '{EXPECTED_PRODUCT_ID}' in order items. "
    f"Found product IDs: {product_ids}"
)
assert assertion_condition_4, assertion_message_4

# Action

**Simulated User**: I recently ordered shoes. I want to exchange that.

**Action Agent**: Could you specify the order ID and give me your name and zip code?

**Simulated User**: My name is Ethan Garcia and my zip code is 80280. I don't know the order ID. I was hoping you could find it on your own through the system.

In [ ]:
import retail

retail.find_user_id_by_name_zip(first_name="Ethan", last_name="Garcia", zip_code="80280")

'ethan_garcia_1261'

In [ ]:
retail.get_user_details(user_id="ethan_garcia_1261")

{'name': {'first_name': 'Ethan', 'last_name': 'Garcia'},
 'address': {'address1': '667 Highland Drive',
  'address2': 'Suite 865',
  'city': 'Denver',
  'country': 'USA',
  'state': 'CO',
  'zip': '80280'},
 'email': 'ethan.garcia8085@example.com',
 'payment_methods': {'paypal_3798357': {'id': 'paypal_3798357',
   'source': 'paypal'},
  'gift_card_4332117': {'id': 'gift_card_4332117',
   'source': 'gift_card',
   'balance': 86}},
 'orders': ['#W4967593', '#W9911714', '#W5733668']}

In [ ]:
retail.get_order_details(order_id="#W4967593")

{'order_id': '#W4967593',
 'user_id': 'ethan_garcia_1261',
 'address': {'address1': '667 Highland Drive',
  'address2': 'Suite 865',
  'city': 'Denver',
  'country': 'USA',
  'state': 'CO',
  'zip': '80280'},
 'items': [{'name': 'Office Chair',
   'product_id': '4794339885',
   'item_id': '4168944673',
   'price': 471.82,
   'options': {'material': 'leather',
    'color': 'blue',
    'armrest': 'none',
    'backrest height': 'standard'}},
  {'name': 'Electric Toothbrush',
   'product_id': '7352963235',
   'item_id': '3320557165',
   'price': 188.67,
   'options': {'color': 'blue',
    'speed settings': 'high',
    'battery type': 'AA batteries'}},
  {'name': 'Backpack',
   'product_id': '2524789262',
   'item_id': '8054888773',
   'price': 206.03,
   'options': {'color': 'grey',
    'size': 'small',
    'material': 'nylon',
    'compartment': 'laptop'}},
  {'name': 'Wristwatch',
   'product_id': '6066914160',
   'item_id': '1355937109',
   'price': 1985.3,
   'options': {'strap materia

In [ ]:
retail.get_order_details(order_id="#W9911714")

{'order_id': '#W9911714',
 'user_id': 'ethan_garcia_1261',
 'address': {'address1': '667 Highland Drive',
  'address2': 'Suite 865',
  'city': 'Denver',
  'country': 'USA',
  'state': 'CO',
  'zip': '80280'},
 'items': [{'name': 'Water Bottle',
   'product_id': '8310926033',
   'item_id': '2366567022',
   'price': 54.04,
   'options': {'capacity': '1000ml',
    'material': 'stainless steel',
    'color': 'blue'}},
  {'name': 'Mechanical Keyboard',
   'product_id': '1656367028',
   'item_id': '1340995114',
   'price': 235.13,
   'options': {'switch type': 'tactile',
    'backlight': 'none',
    'size': 'full size'}},
  {'name': 'Running Shoes',
   'product_id': '6938111410',
   'item_id': '9791469541',
   'price': 147.05,
   'options': {'size': '9',
    'color': 'yellow',
    'material': 'synthetic',
    'sole': 'rubber'}},
  {'name': 'Makeup Kit',
   'product_id': '5149340237',
   'item_id': '1763705424',
   'price': 235.44,
   'options': {'skin tone': 'dark',
    'kit size': 'professi

In [ ]:
retail.get_order_details(order_id="#W5733668")

{'order_id': '#W5733668',
 'user_id': 'ethan_garcia_1261',
 'address': {'address1': '667 Highland Drive',
  'address2': 'Suite 865',
  'city': 'Denver',
  'country': 'USA',
  'state': 'CO',
  'zip': '80280'},
 'items': [{'name': 'Office Chair',
   'product_id': '4794339885',
   'item_id': '8323284863',
   'price': 511.24,
   'options': {'material': 'fabric',
    'color': 'blue',
    'armrest': 'adjustable',
    'backrest height': 'standard'}},
  {'name': 'Portable Charger',
   'product_id': '6942297802',
   'item_id': '7866854614',
   'price': 105.49,
   'options': {'capacity': '5000mAh', 'output': 'USB-C', 'color': 'white'}},
  {'name': 'Electric Kettle',
   'product_id': '1075968781',
   'item_id': '4064702754',
   'price': 159.78,
   'options': {'capacity': '2L', 'material': 'glass', 'color': 'white'}}],
 'fulfillments': [{'tracking_id': ['365054630723'],
   'item_ids': ['8323284863', '7866854614', '4064702754']}],
 'status': 'processed',
 'payment_history': [{'transaction_type': 'p

**Action Agent**: Order #W9911714 contains shoes. What would you like to replace it with?

**Simulated User**: I want to replace it with the item ID 4107812777.

In [ ]:
retail.get_product_details(product_id="6938111410")

{'name': 'Running Shoes',
 'product_id': '6938111410',
 'variants': {'4153505238': {'item_id': '4153505238',
   'options': {'size': '8',
    'color': 'red',
    'material': 'leather',
    'sole': 'EVA'},
   'available': True,
   'price': 158.67},
  '1775591963': {'item_id': '1775591963',
   'options': {'size': '10',
    'color': 'white',
    'material': 'leather',
    'sole': 'EVA'},
   'available': True,
   'price': 154.75},
  '9635758562': {'item_id': '9635758562',
   'options': {'size': '9',
    'color': 'white',
    'material': 'mesh',
    'sole': 'rubber'},
   'available': True,
   'price': 148.95},
  '9791469541': {'item_id': '9791469541',
   'options': {'size': '9',
    'color': 'yellow',
    'material': 'synthetic',
    'sole': 'rubber'},
   'available': True,
   'price': 147.05},
  '4107812777': {'item_id': '4107812777',
   'options': {'size': '9',
    'color': 'black',
    'material': 'synthetic',
    'sole': 'rubber'},
   'available': True,
   'price': 155.33}}}

**Action Agent**: We do have item ID 4107812777 in stock. Would you like to confirm the modification and proceed?

**Simulated User**: Actually, could you also replace the item ID 1656367028 with 1421289881? I think it's the same order.

**Action Agent**: 1656367028 is the product ID. The item ID is 1340995114. I'm assuming you would like to replace 1340995114 with 1421289881.

**Simulated User**: Yes.


In [ ]:
retail.get_product_details(product_id="1656367028")

{'name': 'Mechanical Keyboard',
 'product_id': '1656367028',
 'variants': {'9690244451': {'item_id': '9690244451',
   'options': {'switch type': 'clicky', 'backlight': 'RGB', 'size': '60%'},
   'available': False,
   'price': 236.51},
  '7706410293': {'item_id': '7706410293',
   'options': {'switch type': 'clicky',
    'backlight': 'none',
    'size': 'full size'},
   'available': True,
   'price': 269.16},
  '3616838507': {'item_id': '3616838507',
   'options': {'switch type': 'tactile',
    'backlight': 'white',
    'size': 'full size'},
   'available': True,
   'price': 226.11},
  '8484921793': {'item_id': '8484921793',
   'options': {'switch type': 'linear', 'backlight': 'RGB', 'size': '80%'},
   'available': True,
   'price': 230.15},
  '1340995114': {'item_id': '1340995114',
   'options': {'switch type': 'tactile',
    'backlight': 'none',
    'size': 'full size'},
   'available': False,
   'price': 235.13},
  '6342039236': {'item_id': '6342039236',
   'options': {'switch type': 

In [ ]:
retail.calculate('155.33 - 147.05 + 268.77 - 235.13')

'41.92'

**Action Agent**: To summarize:

- Replace running shoes (item ID 9791469541) with the item ID 4107812777.
- Replace item ID 1340995114 with item ID 1421289881.

You would have to pay additional \$41.92 for these changes. Would you like to confirm and proceed? Also which payment method you would like to use gift card or paypal.

**Simulated User**: Yes, please. Also, link the price differences to my gift card.

In [ ]:
retail.modify_pending_order_items(
    order_id="#W9911714",
    item_ids=["9791469541", "1340995114"],
    new_item_ids=["4107812777", "1421289881"],
    payment_method_id="gift_card_4332117"
)

{'order_id': '#W9911714',
 'user_id': 'ethan_garcia_1261',
 'address': {'address1': '667 Highland Drive',
  'address2': 'Suite 865',
  'city': 'Denver',
  'country': 'USA',
  'state': 'CO',
  'zip': '80280'},
 'items': [{'name': 'Water Bottle',
   'product_id': '8310926033',
   'item_id': '2366567022',
   'price': 54.04,
   'options': {'capacity': '1000ml',
    'material': 'stainless steel',
    'color': 'blue'}},
  {'name': 'Mechanical Keyboard',
   'product_id': '1656367028',
   'item_id': '1421289881',
   'price': 268.77,
   'options': {'switch type': 'linear', 'backlight': 'none', 'size': '80%'}},
  {'name': 'Running Shoes',
   'product_id': '6938111410',
   'item_id': '4107812777',
   'price': 155.33,
   'options': {'size': '9',
    'color': 'black',
    'material': 'synthetic',
    'sole': 'rubber'}},
  {'name': 'Makeup Kit',
   'product_id': '5149340237',
   'item_id': '1763705424',
   'price': 235.44,
   'options': {'skin tone': 'dark',
    'kit size': 'professional',
    'bran

**Action Agent**: Your order has been modified successfully.

Is there anything else I can help you with today?

**Simulated User**: Yes. Could you tell me what my gift card balance is?

In [ ]:
retail.calculate(expression='86 - 41.92')

'44.08'

**Action Agent**: Your initial gift card balance was `86`. With the recent item modifications, the price difference comes out to be `41.92`, since the new items cost more than the original. Hence, your gift card balance will be `44.08`.

**Simulated User**: Cool. Thanks.

**Action Agent**: Is there anything else I can help you with?

# Final Assertion

1. The status of order `#W9911714` is `pending (item modified)`.
2. The order `#W9911714` contains an item with the ID `1421289881`.
3. The order `#W9911714` contains an item with the ID `4107812777`.

In [ ]:
import retail
from Scripts.assertions_utils import *

# --- Constants ---
ORDER_ID = "#W9911714"
EXPECTED_STATUS = "pending (item modified)"
EXPECTED_ITEM_ID_1 = "1421289881"
EXPECTED_ITEM_ID_2 = "4107812777"

# --- Data Gathering ---
order_details = None
api_error = None
try:
    order_details = retail.get_order_details(order_id=ORDER_ID)
except Exception as e:
    api_error = str(e)

# --- Assertion 1: Verify order status ---
actual_status = order_details.get("status") if order_details else None
assertion_condition_1 = compare_strings(actual_status , EXPECTED_STATUS)
assertion_message_1 = (
    f"Assertion 1 Failed: Expected status '{EXPECTED_STATUS}', got '{actual_status}'. API error: {api_error}"
)
assert assertion_condition_1, assertion_message_1

# --- Gather all item IDs in the order ---
items = order_details.get("items", []) if order_details else []
item_ids_in_order = {str(item.get("item_id")) for item in items}

# --- Assertion 2: Item with ID 1421289881 exists ---
assertion_condition_2 = EXPECTED_ITEM_ID_1 in item_ids_in_order
assertion_message_2 = (
    f"Assertion 2 Failed: Expected item ID '{EXPECTED_ITEM_ID_1}' not found in order. Found IDs: {item_ids_in_order}"
)
assert assertion_condition_2, assertion_message_2

# --- Assertion 3: Item with ID 4107812777 exists ---
assertion_condition_3 = EXPECTED_ITEM_ID_2 in item_ids_in_order
assertion_message_3 = (
    f"Assertion 3 Failed: Expected item ID '{EXPECTED_ITEM_ID_2}' not found in order. Found IDs: {item_ids_in_order}"
)
assert assertion_condition_3, assertion_message_3
